# 📝 [제 4편] 머신러닝의 개념과 종류

**학년/반/번호:** 2학년 ( )반 ( )번  **이름:** (           )

> 근거 자료: 박성호, 「과정 소개」 (ML_LEC_01), 「머신러닝(지도학습/비지도학습) 개요」 (ML_LEC_12)

---

## 🎯 학습 목표
1. 인공지능, 머신러닝, 딥러닝의 포함 관계를 설명할 수 있다.
2. 지도학습과 비지도학습의 차이점을 이해한다.
3. 회귀(Regression)와 분류(Classification)를 구분하여 적용할 수 있다.

---

## 🧭 지난 시간 복습
* 제 1 ~ 3편: 미분, 수치미분, 행렬 - 인공지능을 만들 **도구** 를 준비했습니다.
* 오늘은 그 도구로 만들 **머신러닝이 무엇인지** 큰 그림을 봅니다.

[⬅ 이전 : 제 3편 벡터와 행렬 - numpy 로 데이터 다루기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/03_vector_matrix.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 5편 선형 회귀 (1) - 오차와 손실함수 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/05_linear_regression_loss.ipynb)

> 💡 **처음 여는 경우** : 파일 → **Drive에 사본 저장** 을 누르고, 내 사본에서 공부하세요. (빈칸을 채우고 저장할 수 있습니다)
>
> 🧹 **다 본 뒤에는** : 수정 → **모든 출력 지우기** 후 저장하면 애니메이션 때문에 무거워진 사본이 다시 가벼워집니다.

## ⚙️ 0단계. 준비하기
아래 셀을 **가장 먼저 한 번** 실행하세요. (그래프 한글 글꼴과 애니메이션 설정)

In [ ]:
#@title ▶ 준비 셀 (먼저 실행하세요)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib import animation
from IPython.display import HTML, display
import urllib.request, os

font_path = 'NanumGothic.ttf'
try:
    if not os.path.exists(font_path):
        urllib.request.urlretrieve('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', font_path)
    fm.fontManager.addfont(font_path)
    plt.rc('font', family=fm.FontProperties(fname=font_path).get_name())
except Exception as e:
    print('글꼴을 내려받지 못했습니다(그래프의 한글이 깨질 수 있음):', e)
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['animation.embed_limit'] = 50
def show_anim(anim):
    # 애니메이션을 코랩 화면에 재생 버튼과 함께 보여 주기
    display(HTML(anim.to_jshtml()))
np.set_printoptions(precision=6, suppress=True)
print('✅ 준비 완료!')

## 📖 개념 다지기 1. 인공지능 ⊃ 머신러닝 ⊃ 딥러닝

* **인공지능 (AI)** : 인간의 학습·추론 능력을 컴퓨터로 구현하는 가장 넓은 개념
* **머신러닝 (ML)** : 데이터에서 특성과 패턴을 **스스로 학습** 해서, 처음 보는 데이터의 결과를 예측하는 인공지능의 한 분야 (회귀, 분류, 신경망 …)
* **딥러닝 (DL)** : 머신러닝 중에서 뇌를 닮은 **신경망(Neural Network)** 을 깊게 쌓아 학습하는 방법 (CNN, RNN …)

> **[이해해 보기]** 💡 포함 관계: ( 　　　　 ) ⊃ ( 　　　　 ) ⊃ ( 　　　　 )

### 왜 프레임워크 없이 직접 만들어 볼까?
TensorFlow 같은 도구를 쓰면 몇 줄로 인공지능을 만들 수 있지만, 속이 보이지 않는 **블랙박스** 입니다.
이 학습지에서는 **numpy 로 직접 구현** 하면서 수학(미분, 행렬)이 어떻게 인공지능이 되는지 확인합니다.

In [ ]:
#@title 🖼️ 포함 관계 그림
fig, ax = plt.subplots(figsize=(7, 5))
for r, c, name, y in [(2.4, '#dbe9ff', '인공지능 (AI)', 2.0), (1.6, '#c3f0c3', '머신러닝 (ML)', 1.2), (0.85, '#ffd6a8', '딥러닝 (DL)', 0.1)]:
    ax.add_patch(plt.Circle((0, -2.4 + r), r, fc=c, ec='k', lw=1.5))
    ax.text(0, -2.4 + r + y - 0.2 if name != '딥러닝 (DL)' else -2.4 + r, name, ha='center', va='center', fontsize=14, weight='bold')
ax.set_xlim(-2.6, 2.6); ax.set_ylim(-2.6, 2.6); ax.set_aspect('equal'); ax.axis('off'); plt.show()

## 📖 개념 다지기 2. 지도학습 vs 비지도학습

| | 지도학습 (Supervised) | 비지도학습 (Unsupervised) |
|:-|:-|:-|
| 데이터 | 입력 x 와 **정답 t (label)** 이 함께 있다 | **입력만** 있고 정답이 없다 |
| 비유 | 선생님이 문제와 정답을 함께 주며 가르침 | 학생 스스로 비슷한 것끼리 묶어 봄 |
| 목적 | 처음 보는 입력의 정답 **예측** | 숨은 **패턴**·그룹(군집) 발견 |
| 예 | 공부 시간 → 시험 점수, 사진 → 고양이/강아지 | 뉴스 기사 묶기, 비슷한 고객 묶어 상품 추천 |

## 📖 개념 다지기 3. 회귀 vs 분류 (지도학습의 두 종류)

| 구분 | 예측하는 값 | 예 |
|:-:|:-|:-|
| **회귀 (Regression)** | **연속적인 숫자** | 집 평수 → 집 가격, 공부 시간 → 시험 점수 |
| **분류 (Classification)** | **종류 (category)** | 공부 시간 → 합격/불합격, 이메일 → 스팸/정상 |

> **[연습 1]** 회귀와 분류 중 어느 것일까요?
> ① 사진 속 동물이 고양이인지 강아지인지 판별 → ( 　　 )
> ② 오늘의 온도·습도로 내일의 강수량(mm) 예측 → ( 　　 )
> ③ 키와 몸무게로 신발 치수(mm) 예측 → ( 　　 )
> ④ 손글씨 사진이 0 ~ 9 중 어느 숫자인지 판별 → ( 　　 )

In [ ]:
np.random.seed(1)
hours = np.array([9, 14, 21, 27, 32, 37]); score = np.array([74, 81, 86, 90, 88, 92])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
W_, b_ = np.polyfit(hours, score, 1); xs_ = np.array([5, 45])
a1.scatter(hours, score, s=70, color='royalblue'); a1.plot(xs_, W_*xs_ + b_, 'r-', lw=2)
a1.scatter([43], [W_*43 + b_], s=150, color='crimson', marker='*', zorder=5)
a1.text(27, W_*43 + b_ - 8, f'43시간 → 약 {W_*43 + b_:.0f}점?', color='crimson', fontsize=11)
a1.set_title('회귀 : 공부 시간 → 점수 (숫자)', fontsize=13); a1.set_xlabel('공부 시간'); a1.grid(alpha=0.3)
h2 = np.arange(2, 21, 2); r2 = (h2 >= 14).astype(int)
a2.scatter(h2, r2, s=70, c=['navy' if v == 0 else 'crimson' for v in r2]); a2.axvline(13, color='green', ls='--', lw=2)
a2.set_yticks([0, 1]); a2.set_yticklabels(['불합격', '합격']); a2.set_xlabel('공부 시간')
a2.set_title('분류 : 공부 시간 → 합격 / 불합격 (종류)', fontsize=13); a2.grid(alpha=0.3)
plt.show()

## 📖 개념 다지기 4. 머신러닝이 일하는 순서

```
 ① 학습 데이터 입력 (training data)  →  ② 학습 (learning)  →  ③ 처음 보는 데이터로 질문 (test data)  →  ④ 예측 (predict)
```
예) 공부 시간 - 점수 데이터로 학습한 뒤, 데이터에 없는 **43시간** 의 점수를 묻는다.

## 🎬 애니메이션 1. 비지도학습 - 정답 없이 스스로 묶기 (군집화)

아래 점들에는 **정답(색)이 없습니다.** 컴퓨터가 "가까운 것끼리" 기준으로 3개의 그룹을 스스로 찾아가는 과정입니다. (k-평균 군집화)
1. 그룹의 중심(★)을 아무 데나 3개 놓는다.
2. 각 점을 가장 가까운 ★ 의 그룹으로 칠한다.
3. ★ 을 그 그룹 점들의 **평균 위치** 로 옮긴다. → 2, 3 반복

In [ ]:
#@title 🎬 애니메이션 1 실행 (군집화)
np.random.seed(3)
centers_true = np.array([[2, 2], [7, 3], [4, 7]])
P = np.vstack([c + np.random.randn(40, 2)*0.9 for c in centers_true])
C = np.array([[0.0, 9.0], [9.0, 9.0], [5.0, -0.5]])       # 아무 데나 놓은 출발점
states = []
for it in range(8):
    lab = np.argmin(((P[:, None, :] - C[None, :, :])**2).sum(-1), axis=1)
    states.append((C.copy(), lab.copy(), '② 가까운 ★ 로 칠하기'))
    C = np.array([P[lab == k].mean(axis=0) if np.any(lab == k) else C[k] for k in range(3)])
    states.append((C.copy(), lab.copy(), '③ ★ 을 평균 위치로 이동'))
colors = np.array(['crimson', 'royalblue', 'green'])
frames = [0]*6 + [k for k in range(len(states)) for _ in range(4)] + [len(states) - 1]*6
fig, ax = plt.subplots(figsize=(6.5, 6))
def draw(f):
    k = frames[f]; Ck, lab, msg = states[k]
    ax.clear()
    if f < 6:
        ax.scatter(P[:, 0], P[:, 1], s=30, color='gray'); msg = '① 정답(색) 없는 데이터'
    else:
        ax.scatter(P[:, 0], P[:, 1], s=30, c=colors[lab])
    ax.scatter(Ck[:, 0], Ck[:, 1], s=400, marker='*', c=colors, edgecolors='k', linewidths=1.5, zorder=5)
    ax.set_xlim(-1, 10); ax.set_ylim(-1, 10); ax.grid(alpha=0.3)
    ax.set_title(f'{msg}   (반복 {k//2 + 1})' if f >= 6 else msg, fontsize=13)
anim = animation.FuncAnimation(fig, draw, frames=len(frames), interval=200)
plt.close(fig)
show_anim(anim)

> **[관찰 1]** 처음 ★ 들은 모두 위쪽에 몰려 있었지만, 반복할수록 각 ★ 이 점들이 모인 ( 　　　 ) 으로 이동한다.
> 이 과정에서 컴퓨터는 정답을 ( 받았다 / 받지 않았다 ).

## ✏️ 연습 문제

> **[문제 1]** 다음을 지도학습(회귀), 지도학습(분류), 비지도학습 으로 나누세요.
> ① 지난 10년의 아파트 거래 기록으로 새 아파트 가격 예측 ( 　　 )
> ② 구매 기록이 비슷한 손님들을 몇 그룹으로 묶기 ( 　　 )
> ③ 엑스레이 사진으로 폐렴 여부 판단 ( 　　 )

### 🤔 오늘의 질문 (심화 토의)
음악 앱이나 동영상 사이트의 "당신이 좋아할 만한 콘텐츠" 추천은 어떤 방식의 머신러닝에 가까울까요?
(힌트: 정답이 정해져 있을까요? 취향이 비슷한 사람들을 어떻게 모을 수 있을까요?)

(                                                                                                                            )

In [ ]:
#@title 🔑 정답 확인 { display-mode: "form" }
print('포함 관계 : 인공지능 ⊃ 머신러닝 ⊃ 딥러닝')
print('연습 1 : ① 분류  ② 회귀  ③ 회귀  ④ 분류')
print('문제 1 : ① 지도학습(회귀)  ② 비지도학습(군집화)  ③ 지도학습(분류)')

## 📌 정리

| 핵심 | 내용 |
|:---|:---|
| 포함 관계 | 인공지능 ⊃ 머신러닝 ⊃ 딥러닝 |
| 지도학습 | 정답이 있는 데이터로 학습 → 회귀(숫자), 분류(종류) |
| 비지도학습 | 정답 없이 패턴·그룹 찾기 → 군집화 |

### 🏆 도전 과제
회귀의 대표 모델은 **y = Wx + b** 입니다. 데이터에 가장 잘 맞는 W, b 를 찾으려면 "잘 맞는다" 를 숫자로 나타내야 합니다. 어떻게 나타낼 수 있을까요? → 다음 시간: **선형 회귀와 손실함수**

---

[⬅ 이전 : 제 3편 벡터와 행렬 - numpy 로 데이터 다루기](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/03_vector_matrix.ipynb)　|　[📚 목차](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/00_index.ipynb)　|　[다음 : 제 5편 선형 회귀 (1) - 오차와 손실함수 ➡](https://colab.research.google.com/github/chenyenhee/ai-math-colab/blob/python/05_linear_regression_loss.ipynb)